# LC pipeline (TSE -> MNI): One subject, one session test

In [ ]:
import sys
sys.path.insert(0, '..')  

from pathlib import Path
from lc_pipeline import config
from lc_pipeline.data_discovery import build_session_dataframe
from lc_pipeline.brainstem_mask_creation import create_brainstem_mask
from lc_pipeline import step1_n4, step2_t1_to_mni, step3_brainstem_mask
from lc_pipeline import step4_tse_to_t1, step5_hires_grid, step6_resample_tse
from lc_pipeline import step7_masks_to_grid, step8_extract_cr, viz


## Configurations

[!] Edit the `TEST_SUBJECT` / `TEST_SESSION` cell below, then run top to bottom.

In [ ]:
TEST_SUBJECT = "sub002"
TEST_SESSION = "ses001"

USE_FULL_SYN = False
step2_transform = config.STEP2_TRANSFORM_FULL if USE_FULL_SYN else config.STEP2_TRANSFORM_QUICK

OUT_ROOT = config.OUT_ROOT
print('Output root:', OUT_ROOT)
print('Step 2 transform:', step2_transform)


Output root: /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06
Step 2 transform: antsRegistrationSyNQuick[s]


## Data

In [3]:
df = build_session_dataframe(
    session=TEST_SESSION, subject=TEST_SUBJECT,
    root=config.ROOT_NIFTI, output_root=OUT_ROOT,
    modalities=set(config.DEFAULT_KEYWORDS), save=True,
)
df

,subject,session,phase,t1w_path,tse_path
0,sub002,ses001,adaptation,/home/maria/Documents/data/hiwi_sample/tmp-mar...,/home/maria/Documents/data/hiwi_sample/tmp-mar...


In [4]:
row = df.query("subject == @TEST_SUBJECT and session == @TEST_SESSION").iloc[0]
t1w_path = row["t1w_path"]
tse_path = row["tse_path"]
assert t1w_path and t1w_path is not False, "No T1w found for this subject/session"
assert tse_path and tse_path is not False, "No TSE found for this subject/session"
print("T1w:", t1w_path)
print("TSE:", tse_path)

T1w: /home/maria/Documents/data/hiwi_sample/tmp-maria/raw_sample_nifti/sub002/adaptation/slowed_sub002_ses001_slowed_sub002_ses001_19921128/neuropsychology_slowed_1_20250311_185021.100000/anat-T1w_ses-base_acq-mprage_run-01_5_MR/sub002_ses001_anat-T1w_ses-base_acq-mprage_run-01_5.nii.gz
TSE: /home/maria/Documents/data/hiwi_sample/tmp-maria/raw_sample_nifti/sub002/adaptation/slowed_sub002_ses001_slowed_sub002_ses001_19921128/neuropsychology_slowed_1_20250311_185021.100000/anat-TSE_ses-base_acq-lowres_run-01_9_MR/sub002_ses001_anat-TSE_ses-base_acq-lowres_run-01_9.nii.gz


In [5]:
viz.view_raw_inputs(row)  # FSLeyes: raw T1w + TSE over MNI, before any registration

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Brainstem mask in MNI space (built once)

In [5]:
brainstem_mask_mni_path = config.BRAINSTEM_MASK_MNI_PATH
if not brainstem_mask_mni_path.exists():
    print('Creating brainstem mask via Harvard-Oxford atlas (needs FSL on PATH)...')
    create_brainstem_mask(brainstem_mask_mni_path, overwrite=False)
else:
    print('Reusing existing brainstem mask:', brainstem_mask_mni_path)

Reusing existing brainstem mask: /home/maria/Documents/projects/mri_studies/atlases/brainstem_mask_HarvardOxford_MNI_1mm.nii.gz


In [40]:
viz.view_brainstem_mask_mni(brainstem_mask_mni_path)

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 1 -- N4 bias correction

In [6]:
t1w_n4_path = OUT_ROOT / 'step1_n4' / 't1w_n4' / TEST_SUBJECT / TEST_SESSION / 't1w_n4.nii.gz'
t1w_n4_path = step1_n4.n4_bias_correction(input_path=t1w_path, output_path=t1w_n4_path, save=True)
print(t1w_n4_path)

/home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step1_n4/t1w_n4/sub002/ses001/t1w_n4.nii.gz


## Step 2 -- T1w -> MNI (rigid + affine + SyN)

In [7]:
step2_outdir = OUT_ROOT / 'step2_t1w_mni' / TEST_SUBJECT / TEST_SESSION
info_t1_mni = step2_t1_to_mni.register_t1w_n4_to_mni(
    path_fixed_img=config.MNI_TEMPLATE, path_moving_img=t1w_n4_path,
    outdir=step2_outdir, type_of_transform=step2_transform, force=False,
)

Already registered: /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step2_t1w_mni/sub002/ses001/T1w_N4_in_MNI152_0p5mm.nii.gz


In [11]:
viz.view_t1_in_mni(info_t1_mni)

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 3 -- brainstem mask into T1w space

cHANGE 01, The brainstem mask tells ANTs where to evaluate the registration quality


In [8]:
step3_outdir = OUT_ROOT / 'step3_brainstem_t1w' / TEST_SUBJECT / TEST_SESSION
info_brainstem = step3_brainstem_mask.create_brainstem_masks_in_t1w(
    brainstem_mask_mni_path=brainstem_mask_mni_path, t1w_n4_path=t1w_n4_path,
    inverse_transforms=info_t1_mni['invtransforms'], output_dir=step3_outdir, overwrite=False,
)

Brainstem masks already exist.


In [45]:
viz.view_brainstem_in_t1w(t1w_n4_path, info_brainstem, session_label=TEST_SESSION)

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 4 -- TSE -> T1w (rigid + affine, masked, MI)

Change 02 (add mask)

In [9]:
step4_outdir = OUT_ROOT / 'step4_tse_t1w' / TEST_SUBJECT / TEST_SESSION
info_tse_t1w = step4_tse_to_t1.register_tse_to_t1w(
    t1w_n4_path=t1w_n4_path, tse_path=tse_path,
    brainstem_mask_dilated_path=info_brainstem['brainstem_mask_t1w_dilated'],
    output_dir=step4_outdir, overwrite=False,
)

In [42]:
viz.view_tse_in_t1w(t1w_n4_path, info_tse_t1w, info_brainstem, label=f'{TEST_SUBJECT}_{TEST_SESSION}')

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 5 -- high-resolution output grid (built once)

In [10]:
step5_outdir = OUT_ROOT / 'step5_hires_grid'
info_hires_grid = step5_hires_grid.create_mni_brainstem_hires_grid(
    mni_template_path=config.MNI_TEMPLATE, brainstem_mask_mni_path=brainstem_mask_mni_path,
    output_dir=step5_outdir, overwrite=False,
)

FSL directory:        /home/maria/fsl
fslmaths executable:  /home/maria/fsl/share/fsl/bin/fslmaths
MNI template spacing: (0.5, 0.5, 0.5)
MNI template shape:   (364, 436, 364)
Dilating MNI brainstem mask 3 time(s)...
Creating cropped 0.5 mm MNI reference grid...
Full MNI shape:     (364, 436, 364)
Cropped grid shape: (121, 113, 161)
Grid spacing:       (0.5, 0.5, 0.5)
Crop padding:       10 voxels = 5.0 mm
Saved grid:         /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step5_hires_grid/MNI_brainstem_hires_grid_0p5mm.nii.gz


In [11]:
viz.view_hires_grid(info_hires_grid)

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 6 -- composed resample: native TSE -> MNI

In [12]:
step6_outdir = OUT_ROOT / 'step6_tse_mni' / TEST_SUBJECT / TEST_SESSION
info_tse_mni = step6_resample_tse.resample_native_tse_to_mni(
    tse_native_path=tse_path,
    mni_hires_grid_path=info_hires_grid['brainstem_hires_grid'],
    t1_to_mni_warp_path=info_t1_mni['warp01'],
    t1_to_mni_affine_path=info_t1_mni['affine01'],
    tse_to_t1_affine_path=info_tse_t1w['affine_transform'],
    output_dir=step6_outdir, overwrite=False,
)

Applying one composed transform chain:
  TSE native -> T1w -> MNI

Input TSE:        /home/maria/Documents/data/hiwi_sample/tmp-maria/raw_sample_nifti/sub002/adaptation/slowed_sub002_ses001_slowed_sub002_ses001_19921128/neuropsychology_slowed_1_20250311_185021.100000/anat-TSE_ses-base_acq-lowres_run-01_9_MR/sub002_ses001_anat-TSE_ses-base_acq-lowres_run-01_9.nii.gz
Reference grid:   /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step5_hires_grid/MNI_brainstem_hires_grid_0p5mm.nii.gz
TSE -> T1 affine: /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step4_tse_t1w/sub002/ses001/transform02_0GenericAffine.mat
T1 -> MNI affine: /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step2_t1w_mni/sub002/ses001/transform01_0GenericAffine.mat
T1 -> MNI warp:   /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step2_t1w_mni/sub002/ses001/transform01_1Warp.nii.gz
Using double precision for computations.
Default pixel value: 0
Input scalar image: /home/maria/Documents/data/hiw

In [29]:
viz.view_tse_in_mni(info_tse_mni, brainstem_mask_mni_path)

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 7 -- LC and DPT masks onto the tse_in_MNI grid

In [14]:
step7_outdir = OUT_ROOT / 'step7_masks_grid' / TEST_SUBJECT / TEST_SESSION
info_masks_grid = step7_masks_to_grid.put_lc_and_dpt_masks_on_tse_grid(
    lc_mask_mni_path=config.LC_MASK_BOTH, dpt_mask_mni_path=config.DPT_MASK,
    tse_in_mni_path=info_tse_mni['output_tse_mni'], output_dir=step7_outdir, overwrite=False,
)


LC mask
  Source shape:    (364, 436, 364)
  Source spacing:  (0.5, 0.5, 0.5)
  Source origin:   (-90.0, 126.0, -72.0)
  Labels:          [1]
  Already matches: False
  Action:         identity_resampled
  Output shape:   (121, 113, 161)
  Output spacing: (0.5, 0.5, 0.5)
  Output origin:  (-31.0, 59.0, -72.0)
  Output labels:  [1]
  Output voxels:  592
  Saved:          /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step7_masks_grid/sub002/ses001/LC_mask_grid.nii.gz

DPT mask
  Source shape:    (364, 436, 364)
  Source spacing:  (0.5, 0.5, 0.5)
  Source origin:   (-90.0, 126.0, -72.0)
  Labels:          [1]
  Already matches: False
  Action:         identity_resampled
  Output shape:   (121, 113, 161)
  Output spacing: (0.5, 0.5, 0.5)
  Output origin:  (-31.0, 59.0, -72.0)
  Output labels:  [1]
  Output voxels:  1792
  Saved:          /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step7_masks_grid/sub002/ses001/DPT_mask_grid.nii.gz


In [26]:
info_masks_grid["lc_mask_grid"]

'/home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step7_masks_grid/sub002/ses001/LC_mask_grid.nii.gz'

In [43]:
viz.view_masks_on_tse_grid(info_tse_mni, info_masks_grid, label=f'{TEST_SUBJECT}_{TEST_SESSION}')

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

## Step 8 -- peak-slice, per-hemisphere contrast

In [16]:
step8_outdir = OUT_ROOT / 'step8_contrast' / TEST_SUBJECT / TEST_SESSION
summary, per_slice_df = step8_extract_cr.extract_literature_style_peak_contrast(
    subject=TEST_SUBJECT, session=TEST_SESSION,
    tse_in_mni_path=info_tse_mni['output_tse_mni'],
    lc_mask_grid_path=info_masks_grid['lc_mask_grid'],
    dpt_mask_grid_path=info_masks_grid['dpt_mask_grid'],
    output_dir=step8_outdir, overwrite=False,
)
summary

{'literature_style_peak_mean_lr_contrast': 0.2664352968848832,
 'literature_style_peak_z': 89,
 'left_peak_contrast': 0.2514794248939153,
 'left_peak_z': 89,
 'right_peak_contrast': 0.2979230668724999,
 'right_peak_z': 90,
 'subject': 'sub002',
 'session': 'ses001',
 'literature_style_peak_z_mni_mm': -27.5,
 'left_peak_z_mni_mm': -27.5,
 'right_peak_z_mni_mm': -27.0,
 'contrast_formula': '(LC_slice_mean - DPT_slice_mean) / DPT_slice_mean'}

In [17]:
per_slice_df[[
    'z_index', 'z_mni_mm', 'left_contrast', 'right_contrast', 'mean_lr_contrast',
    'left_lc_voxels', 'right_lc_voxels', 'dpt_voxels',
]].dropna(subset=['mean_lr_contrast'])

,z_index,z_mni_mm,left_contrast,right_contrast,mean_lr_contrast,left_lc_voxels,right_lc_voxels,dpt_voxels
88,88,-28.0,0.229506,0.257624,0.243565,10,13,64
89,89,-27.5,0.251479,0.281391,0.266435,11,15,64
90,90,-27.0,0.228103,0.297923,0.263013,12,14,64
91,91,-26.5,0.188705,0.267088,0.227897,15,15,64
92,92,-26.0,0.181276,0.203588,0.192432,16,19,64
93,93,-25.5,0.164813,0.168368,0.166591,16,19,64
94,94,-25.0,0.126312,0.135437,0.130874,17,19,64
95,95,-24.5,0.089838,0.121328,0.105583,15,20,64
96,96,-24.0,0.073155,0.128601,0.100878,15,20,64
97,97,-23.5,0.053054,0.124579,0.088817,15,19,64


In [19]:
per_slice_df.nlargest(
    10,
    "mean_lr_contrast",
)[
    [
        "z_index",
        "z_mni_mm",
        "left_contrast",
        "right_contrast",
        "mean_lr_contrast",
    ]
]

,z_index,z_mni_mm,left_contrast,right_contrast,mean_lr_contrast
89,89,-27.5,0.251479,0.281391,0.266435
90,90,-27.0,0.228103,0.297923,0.263013
88,88,-28.0,0.229506,0.257624,0.243565
91,91,-26.5,0.188705,0.267088,0.227897
92,92,-26.0,0.181276,0.203588,0.192432
93,93,-25.5,0.164813,0.168368,0.166591
100,100,-22.0,0.123077,0.155134,0.139106
103,103,-20.5,0.140626,0.131737,0.136181
94,94,-25.0,0.126312,0.135437,0.130874
99,99,-22.5,0.098441,0.157084,0.127763


In [ ]:
info_masks_grid["dpt_mask_grid"]

'/home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/step7_masks_grid/sub002/ses001/DPT_mask_grid.nii.gz'

In [50]:
import subprocess

INFO_TSE_MNI = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp08/step6_tse_mni/sub002/ses001/tse_in_MNI_brainstem_0p5mm.nii.gz"
LC_MASK_GRID = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp08/step7_masks_grid/sub002/ses001/LC_mask_grid.nii.gz"
DPT_MASK_GRIP = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp08/step7_masks_grid/sub002/ses001/DPT_mask_grid.nii.gz"
label=f'{TEST_SUBJECT}_{TEST_SESSION}'

subprocess.Popen([
        "fsleyes", "--scene", "ortho", "--layout", "horizontal", "--displaySpace", str(INFO_TSE_MNI), "--robustRange",
        str(INFO_TSE_MNI), "--name", f"{label}_TSE_in_MNI", "--cmap", "greyscale", "--alpha", "100",
        str(LC_MASK_GRID), "--name", "LC mask", "--overlayType", "mask",
        "--maskColour", "1", "0", "0", "--threshold", "0.5", "2.5", "--alpha", "100",
        "--outline", "--outlineWidth", "2", "--interpolation", "none",
        str(info_masks_grid["dpt_mask_grid"]), "--name", "DPT mask", "--overlayType", "mask",
        "--maskColour", "0", "0", "1", "--threshold", "0.5", "2.5", "--alpha", "100",
        "--outline", "--outlineWidth", "2", "--interpolation", "none",
    ])

<Popen: returncode: None args: ['fsleyes', '--scene', 'ortho', '--layout', '...>

In [46]:
import ants
import numpy as np
from pathlib import Path


def make_dilated_search_mask(mask_path, output_path, dilation_voxels=2):
    """
    Make a dilated LC search mask.

    If MNI is 0.5 mm isotropic:
        2 voxels ≈ 1 mm dilation
        4 voxels ≈ 2 mm dilation
    """

    mask_path = Path(mask_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    mask_img = ants.image_read(str(mask_path)).clone("float")

    mask_bin_arr = (mask_img.numpy() > 0.5).astype("float32")

    mask_bin = ants.from_numpy(
        mask_bin_arr,
        origin=mask_img.origin,
        spacing=mask_img.spacing,
        direction=mask_img.direction,
    )

    mask_dil = ants.iMath(mask_bin, "MD", dilation_voxels)

    mask_dil_arr = (mask_dil.numpy() > 0.5).astype("float32")

    mask_dil_bin = ants.from_numpy(
        mask_dil_arr,
        origin=mask_img.origin,
        spacing=mask_img.spacing,
        direction=mask_img.direction,
    )

    ants.image_write(mask_dil_bin, str(output_path))

    print("Saved:", output_path)
    print("Original voxels:", int(np.sum(mask_bin_arr > 0)))
    print("Dilated voxels:", int(np.sum(mask_dil_arr > 0)))

    return output_path

In [47]:
LC_LEFT_MASK = Path("/home/maria/Documents/projects/mri_studies/atlases/LCmetaMask_left_MNI05_s01f_plus50.nii.gz")
LC_RIGHT_MASK = Path("/home/maria/Documents/projects/mri_studies/atlases/LCmetaMask_right_MNI05_s01f_plus50.nii.gz")

LC_SEARCH_DIR = OUT_ROOT / "atlases" / "lc_search_masks"

LC_LEFT_SEARCH_MASK = make_dilated_search_mask(
    mask_path=LC_LEFT_MASK,
    output_path=LC_SEARCH_DIR / "LC_left_search_mask_1mm.nii.gz",
    dilation_voxels=2,
)

LC_RIGHT_SEARCH_MASK = make_dilated_search_mask(
    mask_path=LC_RIGHT_MASK,
    output_path=LC_SEARCH_DIR / "LC_right_search_mask_1mm.nii.gz",
    dilation_voxels=2,
)

Saved: /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/atlases/lc_search_masks/LC_left_search_mask_1mm.nii.gz
Original voxels: 322
Dilated voxels: 1652
Saved: /home/maria/Documents/data/hiwi_sample/tmp-maria-exp06/atlases/lc_search_masks/LC_right_search_mask_1mm.nii.gz
Original voxels: 270
Dilated voxels: 1462
